# OHLC data

This file gets OHLC data of securities listed in NEPSE and dumps it in CSV format for further analysis.

In [3]:
from datetime import datetime
import requests
import json
import csv

In [4]:
class NepseData:
    def __init__(self, symbol, date_from, date_to, currency,
                 time_frame) -> None:
        self.symbol = symbol
        self.date_from = self.date_to_unix_timestamp(date_from)
        self.date_to = self.date_to_unix_timestamp(date_to)
        self.currency = currency
        self.time_frame = time_frame
        self.url = f'https://nepsealpha.com/trading/1/history?symbol={self.symbol}&resolution={self.time_frame}&from={self.date_from}&to={self.date_to}&pass=ok&currencyCode={self.currency}'

    def date_to_unix_timestamp(self, date_str):
        date_obj = datetime.strptime(date_str, "%Y-%m-%d")
        return int(date_obj.timestamp())

    def dump_data(self):
        try:
            headers = {
                'User-Agent': 'Mozilla/61.0',
            }

            res = requests.get(self.url, headers=headers)
            if res.status_code == 200:
                ohlc_data = res.json()
                print('Data fetched.')
                if ohlc_data.get("s") != "ok":
                    print("Error: API response is not 'ok'")
                    return
                    
                # Extract OHLC data
                _timestamps = ohlc_data['t']
                timestamps = [datetime.utcfromtimestamp(ts).strftime('%Y-%m-%dT%H:%M:%S') for ts in _timestamps]
                closes = ohlc_data["c"]
                opens = ohlc_data["o"]
                highs = ohlc_data["h"]
                lows = ohlc_data["l"]
                volumes = ohlc_data["v"]
                
                ohlc_tuples = list(zip(timestamps, opens, highs, lows, closes, volumes))
            
                with open(f'data.csv', "w", newline="") as csv_file:
                    csv_writer = csv.writer(csv_file)
                    csv_writer.writerow(["Timestamp", "Open", "High", "Low", "Close", "Volume"])
                    csv_writer.writerows(ohlc_tuples)
                print('Data converted from json to csv.')
            else:
                print(f'API request failed with status code {res.status_code}')
        except requests.exceptions.RequestException as e:
            print(f'Error making API request: {e}')
        except json.JSONDecodeError as e:
            print(f'Error parsing JSON: {e}')